
# 04 - Network Analysis

## Goal

Explore the Enron communication graph produced by the surveillance pipeline.

This notebook focuses on:

- Directed sender-recipient edges
- Node-level network metrics
- Communication hubs
- Betweenness / broker-like actors
- Risk concentration in the network
- Candidate investigation views

This is an exploratory notebook. It does **not** modify pipeline outputs.


## 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

EDGES_PATH = Path("../data/processed/email_network_edges.parquet")
NODES_PATH = Path("../data/processed/email_network_metrics.parquet")
EMAILS_PATH = Path("../data/processed/email_risk_scores.parquet")

edges = pd.read_parquet(EDGES_PATH)
nodes = pd.read_parquet(NODES_PATH)
emails = pd.read_parquet(EMAILS_PATH)

print("Edges:", edges.shape)
print("Nodes:", nodes.shape)
print("Emails:", emails.shape)


## 2. Network Data Overview

The graph is directed:

```text
source → target
```

Each edge represents communication from one sender to one recipient.


In [ ]:
edges.head()

In [ ]:
nodes.head()

In [ ]:
network_summary = pd.DataFrame({
    "metric": [
        "unique_sources",
        "unique_targets",
        "unique_nodes",
        "directed_edges",
        "total_weighted_email_edges",
        "total_risky_edge_emails"
    ],
    "value": [
        edges["source"].nunique(),
        edges["target"].nunique(),
        nodes["node"].nunique(),
        len(edges),
        edges["email_count"].sum(),
        edges["risky_email_count"].sum()
    ]
})

network_summary

## 3. Strongest Communication Relationships

In [ ]:
top_edges = (
    edges.sort_values(["email_count", "risky_email_count"], ascending=False)
    .head(20)
)

top_edges[
    [
        "source",
        "target",
        "email_count",
        "risky_email_count",
        "risky_email_pct",
        "avg_risk_phrase_score",
    ]
]

In [ ]:
plot_edges = top_edges.head(15).copy()
plot_edges["edge_label"] = plot_edges["source"].str[:22] + " → " + plot_edges["target"].str[:22]
plot_edges = plot_edges.sort_values("email_count")

plt.figure(figsize=(11, 6))
plt.barh(plot_edges["edge_label"], plot_edges["email_count"])
plt.title("Top Communication Edges by Email Count")
plt.xlabel("Email count")
plt.ylabel("Sender → Recipient")
plt.tight_layout()
plt.show()


## 4. Most Active Network Nodes

Activity is based on weighted in/out email volume.


In [ ]:
activity_cols = [
    "node",
    "weighted_total_email_count",
    "weighted_in_email_count",
    "weighted_out_email_count",
    "total_connection_count",
    "risky_email_total",
]

nodes.sort_values("weighted_total_email_count", ascending=False)[activity_cols].head(20)

In [ ]:
top_activity = (
    nodes.sort_values("weighted_total_email_count", ascending=False)
    .head(15)
    .sort_values("weighted_total_email_count")
)

plt.figure(figsize=(10, 6))
plt.barh(top_activity["node"], top_activity["weighted_total_email_count"])
plt.title("Top Network Nodes by Weighted Email Volume")
plt.xlabel("Weighted email count")
plt.ylabel("Node")
plt.tight_layout()
plt.show()


## 5. Broker / Bridge Nodes

Betweenness centrality identifies nodes that may sit on communication paths between otherwise separated groups.

In investigation workflows, these can be useful for identifying:

- Information brokers
- Coordination points
- Cross-team communicators


In [ ]:
broker_cols = [
    "node",
    "betweenness_centrality",
    "weighted_total_email_count",
    "total_connection_count",
    "risky_email_total",
]

top_brokers = (
    nodes.sort_values("betweenness_centrality", ascending=False)
    [broker_cols]
    .head(20)
)

top_brokers

In [ ]:
plot_brokers = top_brokers.head(15).sort_values("betweenness_centrality")

plt.figure(figsize=(10, 6))
plt.barh(plot_brokers["node"], plot_brokers["betweenness_centrality"])
plt.title("Top Nodes by Betweenness Centrality")
plt.xlabel("Betweenness centrality")
plt.ylabel("Node")
plt.tight_layout()
plt.show()


## 6. Risk Concentration in the Network

This section identifies nodes that combine high communication activity with elevated risk signals.


In [ ]:
risk_node_cols = [
    "node",
    "risky_email_total",
    "risky_emails_sent",
    "risky_emails_received",
    "weighted_total_email_count",
    "total_connection_count",
    "betweenness_centrality",
]

top_risk_nodes = (
    nodes.sort_values(["risky_email_total", "weighted_total_email_count"], ascending=False)
    [risk_node_cols]
    .head(20)
)

top_risk_nodes

In [ ]:
plot_risk_nodes = top_risk_nodes.head(15).sort_values("risky_email_total")

plt.figure(figsize=(10, 6))
plt.barh(plot_risk_nodes["node"], plot_risk_nodes["risky_email_total"])
plt.title("Top Nodes by Risky Email Total")
plt.xlabel("Risky emails sent + received")
plt.ylabel("Node")
plt.tight_layout()
plt.show()


## 7. Network Activity vs Risk

This scatter plot compares total email volume with risky email total.


In [ ]:
scatter_df = nodes.copy()

plt.figure(figsize=(8, 5))
plt.scatter(
    np.log1p(scatter_df["weighted_total_email_count"]),
    np.log1p(scatter_df["risky_email_total"]),
    alpha=0.35,
    s=12
)
plt.title("Network Activity vs Risky Email Total")
plt.xlabel("log(1 + weighted total email count)")
plt.ylabel("log(1 + risky email total)")
plt.tight_layout()
plt.show()


## 8. Directional Behavior

Compare users who mostly send versus mostly receive communication.


In [ ]:
direction_df = nodes.copy()

direction_df["out_in_ratio"] = (
    (direction_df["weighted_out_email_count"] + 1)
    / (direction_df["weighted_in_email_count"] + 1)
)

direction_df[
    [
        "node",
        "weighted_out_email_count",
        "weighted_in_email_count",
        "out_in_ratio",
        "risky_email_total",
    ]
].sort_values("out_in_ratio", ascending=False).head(20)


## 9. Candidate Investigation Profile

Pick one node and inspect:

- Network metrics
- Top outgoing relationships
- Top incoming relationships
- Top risky emails sent by that node


In [ ]:
# Change this node to inspect another employee/account.
candidate_node = (
    nodes.sort_values("risky_email_total", ascending=False)
    .iloc[0]["node"]
)

candidate_node

In [ ]:
nodes[nodes["node"] == candidate_node].T

In [ ]:
outgoing = (
    edges[edges["source"] == candidate_node]
    .sort_values(["email_count", "risky_email_count"], ascending=False)
    .head(10)
)

outgoing[
    ["source", "target", "email_count", "risky_email_count", "risky_email_pct"]
]

In [ ]:
incoming = (
    edges[edges["target"] == candidate_node]
    .sort_values(["email_count", "risky_email_count"], ascending=False)
    .head(10)
)

incoming[
    ["source", "target", "email_count", "risky_email_count", "risky_email_pct"]
]

In [ ]:
candidate_emails = (
    emails[emails["from_email"] == candidate_node]
    .sort_values("final_risk_score", ascending=False)
)

display_cols = [
    "date",
    "from_email",
    "to_email",
    "subject",
    "final_risk_score",
    "risk_band",
    "risk_phrase_score",
    "risk_phrase_category_count",
]

candidate_emails[[col for col in display_cols if col in candidate_emails.columns]].head(15)


## 10. Key Takeaways

This notebook validates that:

- The Enron graph contains a large directed communication network.
- A small number of actors dominate communication volume.
- Betweenness centrality identifies potential broker-like nodes.
- Risk phrase activity can be aggregated into sender/receiver-level risk views.
- Candidate investigation profiles can be built from network metrics + risky emails.

These outputs directly support the dashboard's Network Analytics and Investigation View pages.
